# Домашнее задание: Спекулятивное декодирование и архитектура Qwen

**Курс:** NLP-2

## Описание задания

В этом домашнем задании мы не будем заниматься обучением моделей с нуля. Вместо этого мы сфокусируемся на **Inference Engineering** — области, которая находится на стыке разработки и исследований и занимается оптимизацией и ускорением работы уже обученных больших языковых моделей (LLM).

Мы реализуем с нуля метод **Speculative Decoding** — один из самых популярных алгоритмических подходов, позволяющий ускорить генерацию текста в 1.5-2.5 раза практически без потери качества. Этот метод используется для оптимизации инференса таких моделей, как Llama, Mixtral и других.

### План работы:

1. **Подготовка окружения**: Настроим среду и загрузим модели.
2. **Реализация Draft модели**: Мы вручную, блок за блоком, соберем архитектуру модели `Qwen2.5-0.5B` (`NanoQwen`). Это позволит нам досконально понять устройство современных LLM, включая `RMSNorm`, `RoPE` и `SwiGLU`.
3. **Реализация цикла спекуляции**: Напишем основной алгоритм, в котором Draft модель быстро генерирует черновик, а Target модель его верифицирует.
4. **Бенчмарк**: Проведем замеры и оценим реальное ускорение, которое дает наш метод.


## Шаг 1: Настройка окружения


In [1]:
# !pip install -q transformers accelerate safetensors sentencepiece

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig, BitsAndBytesConfig, Trainer, TrainingArguments, DataCollatorForLanguageModeling
import numpy as np
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download
import time
import json
import os
from tqdm import tqdm
from peft import LoraConfig, PeftModelForCausalLM
from datasets import Dataset
import torchinfo


# Фиксируем seed для воспроизводимости
torch.manual_seed(42)
np.random.seed(42)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "mps"
    if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Используемое устройство: {device}")

Используемое устройство: cuda


### Загрузка моделей

В качестве целевой модели (`Target`) мы будем использовать `Qwen/Qwen2.5-1.5B-Instruct`. В качестве черновой модели (`Draft`) — `Qwen/Qwen2.5-0.5B-Instruct`. Мы загрузим `Target` с помощью стандартного класса `AutoModelForCausalLM` из `transformers`, а вот `Draft` модель соберем вручную.


In [3]:
TARGET_ID = "Qwen/Qwen2.5-1.5B-Instruct"
DRAFT_ID = "Qwen/Qwen2.5-0.5B-Instruct"

print("Загрузка токенизатора...")
tokenizer = AutoTokenizer.from_pretrained(TARGET_ID)

print("Загрузка Target модели (через AutoModelForCausalLM)...")
# Используем torch_dtype=torch.float16 для экономии VRAM и attn_implementation="sdpa" для использования Flash Attention
target_model = AutoModelForCausalLM.from_pretrained(
    TARGET_ID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa"
)
target_model.eval()
print("Модель Target загружена.")

Загрузка токенизатора...


`torch_dtype` is deprecated! Use `dtype` instead!


Загрузка Target модели (через AutoModelForCausalLM)...
Модель Target загружена.


## Шаг 2: Собираем Draft модель "NanoQwen" (4 балла)

На этом шаге мы не будем использовать `AutoModel`, а соберем модель самостоятельно из отдельных модулей. Это ключевая часть задания, которая поможет понять, как современные трансформеры устроены "под капотом".

Сначала загрузим только конфигурацию `Draft` модели, из которой мы будем брать все параметры архитектуры (размер скрытого слоя, количество голов и т.д.).


In [4]:
draft_config = AutoConfig.from_pretrained(DRAFT_ID)
print("Конфигурация Draft модели:")
print(f"  - Размер скрытого слоя (hidden_size): {draft_config.hidden_size}")
print(f"  - Количество слоев (num_hidden_layers): {draft_config.num_hidden_layers}")
print(
    f"  - Количество голов внимания (num_attention_heads): {draft_config.num_attention_heads}"
)

Конфигурация Draft модели:
  - Размер скрытого слоя (hidden_size): 896
  - Количество слоев (num_hidden_layers): 24
  - Количество голов внимания (num_attention_heads): 14


### Задание 2.1: RMSNorm

Современные модели, такие как Llama и Qwen, используют **Root Mean Square Layer Normalization (RMSNorm)** вместо классического `LayerNorm`. `RMSNorm` проще и вычислительно эффективнее, так как оперирует только масштабированием на основе среднеквадратичного значения и не использует дополнительный сдвиг (bias).

Формула `RMSNorm` для вектора активаций $\mathbf{x}$:
$$ \text{RMS}(\mathbf{x}) = \sqrt{\frac{1}{n} \sum\_{i=1}^{n} x_i^2} $$
$$ \text{RMSNorm}(\mathbf{x}) = \frac{\mathbf{x}}{\text{RMS}(\mathbf{x}) + \epsilon} \cdot \mathbf{w} $$
где $\mathbf{w}$ — обучаемый весовой вектор (гейт), а $\epsilon$ — малая константа для численной стабильности.

**Важно**: При вычислениях в `float16`, промежуточный расчет `variance` (степень `pow(2)`) может привести к переполнению. Поэтому стандартная практика — временно повышать тип данных до `float32` для этого расчета.

**Задание**: Реализуйте `forward` для `QwenRMSNorm`.

**Полезные ссылки**:

- [Root Mean Square Layer Normalization (Zhang and Sennrich, 2019)](https://arxiv.org/abs/1910.07467)


In [5]:
class QwenRMSNorm(nn.Module):
    """
    Реализация Root Mean Square Layer Normalization.

    Аргументы:
        hidden_size (int): Размер скрытого слоя.
        eps (float): Малая константа для предотвращения деления на ноль.
    """

    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        # --- НАЧАЛО ВАШЕГО КОДА ---
        # 1. Запомните исходный тип данных (например, float16).
        # 2. Переведите hidden_states в float32 для стабильных вычислений.
        # 3. Вычислите variance: среднее от квадратов элементов по последней оси.
        #    Не забудьте `keepdim=True`.
        # 4. Нормализуйте hidden_states (torch.rsqrt в помощь)
        # 5. Умножьте на обучаемый вес и верните результат в исходном типе данных.
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)
        # --- КОНЕЦ ВАШЕГО КОДА ---

In [6]:
print("--- Запуск тестов для RMSNorm ---")
try:
    hidden_size = 128
    norm_layer = QwenRMSNorm(hidden_size).to(device).half()

    # Тест 1: Проверка размерности
    dummy_input = torch.randn(2, 10, hidden_size, device=device).half()
    output = norm_layer(dummy_input)
    assert output.shape == dummy_input.shape, (
        f"Ошибка размерности: ожидалось {dummy_input.shape}, получено {output.shape}"
    )
    print("✅ [1/3] Тест на размерность пройден.")

    # Тест 2: Проверка типа данных
    assert output.dtype == torch.float16, (
        f"Ошибка типа данных: ожидалось float16, получено {output.dtype}"
    )
    print("✅ [2/3] Тест на тип данных пройден.")

    # Тест 3: Проверка на NaN
    assert not torch.isnan(output).any(), "В выходе RMSNorm обнаружены NaN значения."
    print("✅ [3/3] Тест на NaN пройден.")

    print("\n🎉 Все тесты для RMSNorm пройдены!")

except Exception as e:
    print(f"❌ Тест RMSNorm провален: {e}")

--- Запуск тестов для RMSNorm ---
✅ [1/3] Тест на размерность пройден.
✅ [2/3] Тест на тип данных пройден.
✅ [3/3] Тест на NaN пройден.

🎉 Все тесты для RMSNorm пройдены!


### Задание 2.2: Rotary Positional Embeddings (RoPE)

`RoPE` — это элегантный способ внедрения позиционной информации, который вместо добавления векторов (как в `sin/cos embeddings`) "вращает" векторы запросов (`Query`) и ключей (`Key`) на угол, зависящий от их позиции.

#### Два подхода к реализации

Существует два эквивалентных способа реализации этого вращения.

1. **Разбиение на пары (Pairwise Rotation)**. Этот метод мы рассматривали на лекции. Вектор признаков $\mathbf{x} = (x_1, x_2, \dots, x_d)$ рассматривается как набор двумерных векторов $(x_{2i-1}, x_{2i})$. Каждый такой вектор вращается в 2D-плоскости:

   $$
   \begin{pmatrix} x'_{2i-1} \\ x'_{2i} \end{pmatrix} =
   \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}
   \begin{pmatrix} x_{2i-1} \\ x_{2i} \end{pmatrix}
   $$

   где $m$ — позиция токена, а $\theta_i$ — частота вращения.

2. **Вращение через половины (`rotate_half`)**. Этот метод используется в реализациях Llama и Qwen, и именно его мы будем использовать. Здесь вектор $\mathbf{x}$ делится на две половины: $\mathbf{x}_1 = (x_1, \dots, x_{d/2})$ и $\mathbf{x}_2 = (x_{d/2+1}, \dots, x_d)$. Вращение реализуется следующим образом:

   $$
   \mathbf{x}'\_m = \mathbf{x}\_m \cos(m\theta) + \text{rotate_half}(\mathbf{x}\_m) \sin(m\theta)
   $$
где операция `rotate_half` преобразует вектор $\mathbf{x} = (\mathbf{x}_1, \mathbf{x}_2)$ в $(-\mathbf{x}_2, \mathbf{x}_1)$. Этот подход легче векторизуется и более эффективен в `torch`.

**Задание**: Реализуйте функцию `apply_rope`, следуя второму подходу.

**Полезные ссылки**:

- [RoFormer: Enhanced Transformer with Rotary Position Embedding (Su et al., 2021)](https://arxiv.org/abs/2104.09864)
- [Подробное объяснение RoPE в блоге EleutherAI](https://blog.eleuther.ai/rotary-embeddings/)


In [7]:
def precompute_freqs_cis(dim: int, end: int, theta: float = 1000000.0):
    """
    Предварительно вычисляет частоты для RoPE в комплексном виде (cos + i*sin).

    Эта функция готовит таблицы синусов и косинусов для всех возможных позиций.
    """
    freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim))
    t = torch.arange(end, device=freqs.device, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    # В реализации Llama/Qwen частоты дублируются для обеих половин
    # Мы вернем косинусы и синусы отдельно для удобства
    freqs_cat = torch.cat((freqs, freqs), dim=-1)
    return torch.cos(freqs_cat), torch.sin(freqs_cat)


def rotate_half(x):
    """Вращает половину скрытых измерений входа."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rope(q, k, cos, sin):
    """
    Применяет RoPE к векторам q и k.
    """
    # --- НАЧАЛО ВАШЕГО КОДА ---
    _dtype = q.dtype
    # 1. Измените размерность cos и sin для бродкастинга с (q, k).
    #    Они должны стать [1, seq_len, 1, head_dim].
    # cos = cos.unsqueeze(0).unsqueeze(2)
    # sin = sin.unsqueeze(0).unsqueeze(2)
    cos = cos.unsqueeze(1)
    sin = sin.unsqueeze(1)
    # 2. Примените формулу вращения к q и k, используя rotate_half.
    #    Операции с float32 (cos/sin) приведут к результату float32.
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    # 3. Верните q_embed и k_embed.
    #    ВАЖНО: Явно приводим к типу q.dtype (float16), иначе упадет assert или следующий слой
    return q_embed.to(_dtype), k_embed.to(_dtype)
    # --- КОНЕЦ ВАШЕГО КОДА ---

In [8]:
print("--- Запуск тестов для RoPE ---")
try:
    head_dim = 64
    seq_len = 10
    batch_size = 2
    num_heads = 4

    xq = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device).half()
    xk = torch.randn(batch_size, seq_len, num_heads, head_dim, device=device).half()
    cos, sin = precompute_freqs_cis(head_dim, seq_len * 2)
    cos, sin = cos.to(device), sin.to(device)

    # Вызываем функцию
    xq_rot, xk_rot = apply_rope(xq, xk, cos[:seq_len], sin[:seq_len])

    # Тест 1: Проверка размерности
    assert xq_rot.shape == xq.shape, (
        f"Ошибка размерности Query: ожидалось {xq.shape}, получено {xq_rot.shape}"
    )
    assert xk_rot.shape == xk.shape, (
        f"Ошибка размерности Key: ожидалось {xk.shape}, получено {xk_rot.shape}"
    )
    print("✅ [1/2] Тест на размерность пройден.")

    # Тест 2: Проверка типа данных
    assert xq_rot.dtype == xq.dtype, (
        f"Ошибка типа данных Query: ожидалось {xq.dtype}, получено {xq_rot.dtype}"
    )
    assert xk_rot.dtype == xk.dtype, (
        f"Ошибка типа данных Key: ожидалось {xk.dtype}, получено {xk_rot.dtype}"
    )
    print("✅ [2/2] Тест на тип данных пройден.")

    print("\n🎉 Все тесты для RoPE пройдены!")
except Exception as e:
    print(f"❌ Тест RoPE провален: {e}")

--- Запуск тестов для RoPE ---
✅ [1/2] Тест на размерность пройден.
✅ [2/2] Тест на тип данных пройден.

🎉 Все тесты для RoPE пройдены!


### Задание 2.3: SwiGLU MLP

Вместо стандартного `FeedForward` блока с одной `ReLU` активацией, современные модели используют `Gated Linear Units (GLU)` и их варианты. В Qwen/Llama используется **SwiGLU**.

Идея состоит в том, чтобы использовать гейт (шлюз) для управления информационным потоком. Входной вектор `x` проецируется двумя разными линейными слоями (`up` и `gate`). Результат `gate` проекции проходит через активацию `SiLU` (также известную как Swish), а затем поэлементно умножается на результат `up` проекции. Это позволяет сети динамически решать, какая информация должна пройти дальше.

Формула `SwiGLU`:
$$ \text{SwiGLU}(x, W*{up}, W*{gate}, W*{down}) = (\text{SiLU}(x W*{gate}) \otimes (x W*{up})) W*{down} $$
где $\otimes$ — поэлементное умножение, а `SiLU` (Swish activation) определяется как:
$$ \text{SiLU}(x) = x \cdot \sigma(x) $$
где $\sigma$ — это сигмоида.

**Задание**: Реализуйте `forward` для `QwenMLP`.

**Полезные ссылки**:

- [GLU Variants Improve Transformer (Shazeer, 2020)](https://arxiv.org/abs/2002.05202)


In [9]:
class QwenMLP(nn.Module):
    """
    Реализация SwiGLU Feed-Forward сети.
    """

    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)

    def forward(self, x):
        # --- НАЧАЛО ВАШЕГО КОДА ---
        # 1. Примените gate_proj и up_proj к входу x.
        # 2. Примените активацию SiLU (F.silu) к выходу gate_proj.
        # 3. Поэлементно перемножьте результат шага 2 и выход up_proj.
        # 4. Пропустите результат через down_proj и верните его.
        down_proj = self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))
        return down_proj
        # --- КОНЕЦ ВАШЕГО КОДА ---

In [10]:
print("--- Запуск тестов для SwiGLU MLP ---")
try:
    # Создаем mock-конфиг для теста
    class MockConfig:
        hidden_size = 128
        intermediate_size = 384

    config = MockConfig()
    mlp = QwenMLP(config).to(device).half()

    x = torch.randn(2, 10, config.hidden_size, device=device).half()
    output = mlp(x)

    # Тест 1: Проверка размерности
    assert output.shape == x.shape, (
        f"Ошибка размерности: ожидалось {x.shape}, получено {output.shape}"
    )
    print("✅ [1/2] Тест на размерность пройден.")

    # Тест 2: Проверка на NaN
    assert not torch.isnan(output).any(), "В выходе MLP обнаружены NaN значения."
    print("✅ [2/2] Тест на NaN пройден.")

    print("\n🎉 Все тесты для SwiGLU MLP пройдены!")
except Exception as e:
    print(f"❌ Тест SwiGLU MLP провален: {e}")

--- Запуск тестов для SwiGLU MLP ---
✅ [1/2] Тест на размерность пройден.
✅ [2/2] Тест на NaN пройден.

🎉 Все тесты для SwiGLU MLP пройдены!


### Задание 2.4: Сборка итоговой модели NanoQwen

Теперь, когда у нас есть все строительные блоки, мы можем собрать из них полноценный слой трансформера (`NanoQwenBlock`) и саму модель (`NanoQwen`).

Вам необходимо дополнить класс `NanoQwenBlock`, правильно соединив все модули. Обратите внимание на порядок операций в трансформерах семейства Llama/Qwen:

1. **Pre-normalization** перед Self-Attention.
2. **Residual Connection** после Self-Attention.
3. **Pre-normalization** перед MLP.
4. **Residual Connection** после MLP.


In [11]:
class QwenAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = config.hidden_size // config.num_attention_heads
        self.num_key_value_heads = config.num_key_value_heads

        self.q_proj = nn.Linear(
            self.hidden_size, self.num_heads * self.head_dim, bias=True
        )
        self.k_proj = nn.Linear(
            self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True
        )
        self.v_proj = nn.Linear(
            self.hidden_size, self.num_key_value_heads * self.head_dim, bias=True
        )
        self.o_proj = nn.Linear(
            self.num_heads * self.head_dim, self.hidden_size, bias=False
        )

    def forward(self, hidden_states, cos, sin):
        bsz, q_len, _ = hidden_states.size()

        q = self.q_proj(hidden_states).view(bsz, q_len, self.num_heads, self.head_dim)
        k = self.k_proj(hidden_states).view(
            bsz, q_len, self.num_key_value_heads, self.head_dim
        )
        v = self.v_proj(hidden_states).view(
            bsz, q_len, self.num_key_value_heads, self.head_dim
        )

        q, k = apply_rope(q, k, cos, sin)

        # Grouped Query Attention (GQA)
        if self.num_key_value_heads != self.num_heads:
            k = k.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=2)
            v = v.repeat_interleave(self.num_heads // self.num_key_value_heads, dim=2)

        q = q.transpose(1, 2)
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Используем встроенную реализацию Flash Attention
        output = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        output = output.transpose(1, 2).contiguous().view(bsz, q_len, -1)
        return self.o_proj(output)


class NanoQwenBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.self_attn = QwenAttention(config)
        self.mlp = QwenMLP(config)
        self.input_layernorm = QwenRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.post_attention_layernorm = QwenRMSNorm(
            config.hidden_size, eps=config.rms_norm_eps
        )

    def forward(self, hidden_states, cos, sin):
        # --- НАЧАЛО ВАШЕГО КОДА ---
        # 1. Pre-normalization и Self-Attention
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, cos, sin)
        hidden_states = residual + hidden_states

        # 2. Pre-normalization и MLP
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states

        # --- КОНЕЦ ВАШЕГО КОДА ---
        return hidden_states


class NanoQwen(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList(
            [NanoQwenBlock(config) for _ in range(config.num_hidden_layers)]
        )
        self.norm = QwenRMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # Предвычисляем RoPE
        self.cos, self.sin = precompute_freqs_cis(
            config.hidden_size // config.num_attention_heads,
            config.max_position_embeddings * 2,
        )
        self.cos = self.cos.to(device)
        self.sin = self.sin.to(device)

    def forward(self, input_ids, attention_mask=None):
        # print(f"[DEBUG]: input shape={input_ids.shape}")
        x = self.embed_tokens(input_ids)
        seq_len = x.shape[1]

        # Выбираем нужный срез из таблицы RoPE
        cos_t = self.cos[:seq_len]
        sin_t = self.sin[:seq_len]

        for layer in self.layers:
            x = layer(x, cos_t, sin_t)

        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

### Загрузка весов

Теперь нам нужно загрузить веса из официального репозитория в нашу самописную модель. Мы уже реализовали функцию `load_weights_manual`, которая делает это и корректно обрабатывает различия в именовании слоев.


In [12]:
def load_weights_manual(model, model_id):
    """
    Загружает веса из файла .safetensors в ручном режиме,
    корректируя имена ключей.
    """
    print("Скачивание весов...")
    file_path = hf_hub_download(repo_id=model_id, filename="model.safetensors")

    print("Загрузка файла safetensors...")
    state_dict = load_file(
        file_path, device="cpu"
    )  # Грузим на CPU, чтобы не занимать VRAM

    new_state_dict = {}
    for key, tensor in state_dict.items():
        new_key = key

        # Если ключ начинается с "model.", удаляем этот префикс
        if key.startswith("model."):
            new_key = key[len("model.") :]

        new_state_dict[new_key] = tensor

    # В некоторых моделях lm_head и embed_tokens используют общие веса (tied weights).
    # Если lm_head нет, копируем его из embed_tokens.
    if (
        "lm_head.weight" not in new_state_dict
        and "embed_tokens.weight" in new_state_dict
    ):
        print("Связывание весов: lm_head.weight <- embed_tokens.weight")
        new_state_dict["lm_head.weight"] = new_state_dict["embed_tokens.weight"]

    print("Загрузка весов в модель...")
    missing, unexpected = model.load_state_dict(new_state_dict, strict=False)

    if not missing and not unexpected:
        print("✅ Веса успешно загружены!")
    else:
        print("❌ Ошибка при загрузке весов:")
        if missing:
            print(f"  Не найдены ключи в state_dict: {missing[:5]}")
        if unexpected:
            print(f"  Лишние ключи в state_dict: {unexpected[:5]}")

    model.to(device).to(torch.float16)
    return model, missing, unexpected

In [13]:
# Инициализация и загрузка Draft модели
draft_model = NanoQwen(draft_config)
draft_model, missing_keys, unexpected_keys = load_weights_manual(draft_model, DRAFT_ID)
draft_model.to(device).eval()

# Тест
print("\n--- Запуск теста для загрузки весов ---")
try:
    assert not missing_keys, f"Найдены недостающие ключи: {missing_keys[:5]}"
    assert not unexpected_keys, f"Найдены лишние ключи: {unexpected_keys[:5]}"
    print("🎉 Тест на загрузку весов пройден!")
except Exception as e:
    print(f"❌ Тест провален: {e}")

Скачивание весов...
Загрузка файла safetensors...
Связывание весов: lm_head.weight <- embed_tokens.weight
Загрузка весов в модель...
✅ Веса успешно загружены!

--- Запуск теста для загрузки весов ---
🎉 Тест на загрузку весов пройден!


### Проверка работоспособности

Давайте убедимся, что наша самописная модель генерирует осмысленный текст. Мы используем простой цикл жадной генерации (`greedy search`).


In [14]:
def generate_simple(model, text, max_new=10):
    """Простая функция для жадной генерации."""
    inputs = tokenizer(text, return_tensors="pt").to(device)
    input_ids = inputs.input_ids

    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        for _ in range(max_new):
            logits = model(input_ids)
            logits = logits.logits if hasattr(logits, 'logits') else logits
            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            input_ids = torch.cat([input_ids, next_token], dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Who are you?"},
]
text = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)

print("--- Ответ от NanoQwen: ---")
print(generate_simple(draft_model, text, max_new=20))

--- Ответ от NanoQwen: ---
system
You are a helpful assistant.
user
Who are you?
assistant
I am a large language model created by Alibaba Cloud. I am called Qwen.


## Шаг 3: Алгоритм спекулятивного декодирования (3 балла)

Теперь самая важная часть. Мы реализуем **Greedy Speculative Decoding**.

**Алгоритм**:

1. **Черновик (Draft)**: Генерируем `K` токенов с помощью быстрой маленькой модели (`Draft`).
2. **Верификация (Verify)**: Прогоняем всю последовательность (префикс + `K` токенов) через медленную большую модель (`Target`) **за один `forward` вызов**.
3. **Проверка (Accept/Reject)**: Сравниваем токены, предсказанные `Target` моделью, с токенами, сгенерированными `Draft` моделью.
   - Находим первый индекс `i`, где предсказания не совпали.
   - Все `i` совпавших токенов считаются "принятыми" и добавляются к результату.
   - В качестве следующего токена мы берем "правильный" токен от `Target` модели на позиции `i`.
   - Все остальные токены из черновика отбрасываются.
4. Повторяем цикл.

**Задание**: Реализуйте логику проверки и принятия токенов.

**Полезные ссылки**:

- [Fast Inference from Transformers via Speculative Decoding (Leviathan et al., 2022)](https://arxiv.org/abs/2211.17192)


In [15]:
def speculative_sampling(
    prefix_text, max_new_tokens, target_model, draft_model, tokenizer, K=5
):
    """
    Реализует цикл спекулятивного декодирования.
    """
    # Форматируем промпт
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prefix_text},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(device)

    generated_ids = input_ids.clone()
    finished_len = input_ids.shape[1] + max_new_tokens

    stats = {"target_calls": 0, "total_accepted": 0, "total_drafted": 0}

    with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        while generated_ids.shape[1] < finished_len:
            prefix_len = generated_ids.shape[1]

            # 1. DRAFT
            draft_ids = generated_ids
            for _ in range(K):
                outputs = draft_model(draft_ids)
                # Добавил для совместимости с HF т.к. там возвращают ModelOutput, а не тензор
                # Извлекаем логиты: если ModelOutput, используем .logits, иначе - это уже тензор
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                draft_ids = torch.cat([draft_ids, next_token], dim=1)
                if next_token.item() == tokenizer.eos_token_id:
                    break

            drafted_tokens = draft_ids[0, prefix_len:]
            if not len(drafted_tokens):
                break  # Если ничего не сгенерировали
            stats["total_drafted"] += len(drafted_tokens)

            # 2. VERIFY
            target_outputs = target_model(draft_ids)
            stats["target_calls"] += 1
            target_logits = target_outputs.logits
            # Нас интересуют предсказания Target модели для токенов, которые сгенерировал Draft
            relevant_logits = target_logits[0, prefix_len - 1 : -1]
            target_preds = torch.argmax(relevant_logits, dim=-1)

            # 3. ACCEPT/REJECT Logic
            n_accepted = 0

            # --- НАЧАЛО ВАШЕГО КОДА ---
            # Проитерируйтесь по `drafted_tokens` и `target_preds`.
            for i in range(len(drafted_tokens)):
                # Увеличивайте `n_accepted`, пока токены совпадают.
                if drafted_tokens[i] != target_preds[i]:
                    # Прервите цикл, как только найдете первое несовпадение.
                    break
                n_accepted += 1
            # --- КОНЕЦ ВАШЕГО КОДА ---

            stats["total_accepted"] += n_accepted

            # Принимаем все совпавшие токены
            accepted_ids = drafted_tokens[:n_accepted]
            generated_ids = torch.cat([generated_ids, accepted_ids.unsqueeze(0)], dim=1)

            # Если достигли лимита, выходим
            if generated_ids.shape[1] >= finished_len:
                break
            
            # Если последний токен - EOS, выходим
            if len(accepted_ids) > 0 and accepted_ids[-1] == tokenizer.eos_token_id:
                break

            # Добавляем один "исправленный" токен от Target модели
            if n_accepted < len(target_preds):
                correct_token = target_preds[n_accepted].view(1, 1)
            else:  # Если все совпало, берем следующий токен от Target модели
                last_logits = target_logits[0, -1, :]
                correct_token = torch.argmax(last_logits).view(1, 1)

            generated_ids = torch.cat([generated_ids, correct_token], dim=1)

            if correct_token.item() == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated_ids[0], skip_special_tokens=True), stats

In [16]:
print("--- Запуск теста для Speculative Decoding ---")
try:
    # Создаем "игрушечные" модели для теста
    class MockModel(nn.Module):
        def __init__(self, vocab_size, response_sequence):
            super().__init__()
            self.vocab_size = vocab_size
            self.response = response_sequence
            self.call_idx = 0

        def forward(self, input_ids):
            if input_ids is None:
                return None  # property hack fix
            batch, seq_len = input_ids.shape
            next_token_idx = min(self.call_idx, len(self.response) - 1)
            next_token = self.response[next_token_idx]
            self.call_idx += 1
            logits = torch.full(
                (batch, seq_len, self.vocab_size), -100.0, device=device
            )
            logits[:, -1, next_token] = 100.0
            return logits

    mock_draft = MockModel(100, [10, 20, 30, 40, 50])
    mock_target = MockModel(100, [10, 20, 99, 40, 50])
    # Хамский хак для property, но в тесте мы вызываем forward напрямую через target(draft_ids)

    def mock_speculative_sampling(target, draft, K=5):
        generated_ids = torch.tensor([[1, 2]], device=device)
        prefix_len = generated_ids.shape[1]
        draft_ids = torch.tensor([[1, 2, 10, 20, 30, 40, 50]], device=device)

        drafted_tokens = draft_ids[0, prefix_len:]
        print(f"Drafted tokens: {drafted_tokens.tolist()}")

        # Эмулируем вызов target модели
        # Важно: MockModel в нашем тесте очень тупая, она просто возвращает следующий токен из списка response
        # независимо от входа. Нам нужно вызвать ее для каждого токена последовательности,
        # чтобы собрать "правильные" предикты.

        # Перепишем логику сбора предиктов для MockModel, чтобы она работала как авторегрессия

        target_preds_list = []

        # Мы хотим проверить предикты для:
        # input=[1, 2] -> предсказание (должно быть 10)
        # input=[... 10] -> предсказание (должно быть 20)
        # input=[... 20] -> предсказание (должно быть 99)

        # Сбросим состояние
        target.call_idx = 0

        # В реальной жизни мы делаем один forward pass.
        # Но наша MockModel возвращает только последний токен.
        # Поэтому для теста мы просто возьмем response_sequence учителя.
        # Это упрощение теста, но оно валидирует логику accept/reject.

        teacher_sequence = target.response  # [10, 20, 99, 40, 50]
        target_preds = torch.tensor(teacher_sequence, device=device)

        print(f"Target preds (ideal): {target_preds.tolist()}")

        n_accepted = 0
        # --- КОПИЯ ВАШЕЙ ЛОГИКИ ИЗ speculative_sampling ---
        # raise NotImplementedError("КОПИЯ ВАШЕЙ ЛОГИКИ ИЗ speculative_sampling")
        for i in range(len(drafted_tokens)):
            # Увеличивайте `n_accepted`, пока токены совпадают.
            if drafted_tokens[i] == target_preds[i]:
                n_accepted += 1
                # Прервите цикл, как только найдете первое несовпадение.
            else:
                break
        # --------------------------------------------------
        return n_accepted

    n_accepted = mock_speculative_sampling(mock_target, mock_draft)
    print(f"Accepted: {n_accepted}")

    assert n_accepted == 2, (
        f"Ошибка в логике Accept/Reject: ожидалось 2 принятых токена, получено {n_accepted}"
    )
    print("🎉 Тест для логики Accept/Reject пройден!")

except Exception as e:
    print(f"❌ Тест провален: {e}")

--- Запуск теста для Speculative Decoding ---
Drafted tokens: [10, 20, 30, 40, 50]
Target preds (ideal): [10, 20, 99, 40, 50]
Accepted: 2
🎉 Тест для логики Accept/Reject пройден!


## Шаг 4: Бенчмарк

Чтобы увидеть реальный выигрыш от спекулятивного декодирования, нам нужно симулировать ситуацию, когда `Target` модель работает значительно медленнее, чем `Draft`. В реальной жизни так и происходит: `Draft` может быть модель на 0.5B параметров, а `Target` — на 70B.

Мы создадим обертку `HeavyTarget`, которая будет искусственно добавлять задержку перед каждым вызовом `forward`, имитируя медленный инференс большой модели. Затем мы сравним время генерации стандартным (авторегрессионным) способом и с помощью нашего спекулятивного алгоритма.Ожидается, что спекулятивное декодирование покажет значительное ускорение (speedup ~ 1.5x+).

Однако, обратите внимание, что для действительно эффективной работы на длинных контекстах нам необходим KV-cache, чтобы заново не пересчитывать уже выполненные вычисления. Именно так speculative decoding реализован в популярных фреймворках.


In [17]:
import pandas as pd


class HeavyTarget:
    def __init__(self, model, delay=0.05):
        self.model = model
        self.delay = delay

    def __call__(self, *args, **kwargs):
        time.sleep(self.delay)
        return self.model(*args, **kwargs)

    @property
    def config(self):
        return self.model.config


def autoregressive(model, text, max_new=50):
    ids = tokenizer(text, return_tensors="pt").to(device).input_ids
    start = time.time()
    cnt = 0
    with torch.no_grad():
        for _ in range(max_new):
            out = model(ids)
            tok = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            ids = torch.cat([ids, tok], dim=1)
            cnt += 1
            if tok.item() == tokenizer.eos_token_id:
                break
    return cnt, time.time() - start


def benchmark_speculative_sampling(
    draft_model, target_model, tokenizer,
    results, 
    prefix_text = "",

):
    prompts = [
        "Write a Python function to calculate Fibonacci numbers.",
        "The capital of France is",
        "Explain the theory of relativity in simple terms.",
    ]
    heavy_target = HeavyTarget(target_model, 0.05)

    print("Running Benchmark..." + f" {prefix_text}" if prefix_text else "")
    for p in prompts:
        # Std
        s_tok, s_time = autoregressive(heavy_target, p)
        s_speed = s_tok / s_time

        # Spec
        start = time.time()
        _, stats = speculative_sampling(p, 50, heavy_target, draft_model, tokenizer)
        spec_time = time.time() - start
        spec_tok = stats["total_accepted"] + stats["target_calls"]
        spec_speed = spec_tok / spec_time

        results.append(
            {
                "Prefix": prefix_text,
                "Prompt": p[:20],
                "Std (t/s)": s_speed,
                "Spec (t/s)": spec_speed,
                "Speedup": spec_speed / s_speed,
                "Acceptance Rate": stats["total_accepted"] / stats["total_drafted"],
                "Accepted": stats["total_accepted"],
                "Drafted": stats["total_drafted"]
            }
        )

    print(pd.DataFrame(results).to_string(float_format="{:.2f}".format))
    return results

results = []
results = benchmark_speculative_sampling(draft_model, target_model, tokenizer, results, prefix_text="NanoQwen")

Running Benchmark... NanoQwen
     Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0  NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1  NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2  NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65


In [18]:
# Для сравнения проверю модель из HF
draft_model_hf = AutoModelForCausalLM.from_pretrained(
    DRAFT_ID, torch_dtype=torch.float16, device_map="auto", attn_implementation="sdpa"
)
draft_model_hf.eval()

results = benchmark_speculative_sampling(draft_model_hf, target_model, tokenizer, results, prefix_text="NanoQwen_HF")


Running Benchmark... NanoQwen_HF
        Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0     NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1     NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2     NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3  NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4  NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5  NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65


#### Вывод

Удалось реализовать модель, показывающую производительность аналогичную
(и даже чуть выше) оригинальной модели из Transformers.

## Что дальше?

Чтобы получить максимальный балл за ДЗ, попробуйте реализовать одну из следующих идей:

### 1. Quantized Draft Model (0.5 балла)

Мы ускорили модель алгоритмически. А давайте теперь ускорим Draft модель аппаратно! Попробуйте квантовать `NanoQwen` в 4-бит (используя библиотеки `bitsandbytes` или `GPTQ`) и посмотрите, как изменится время генерации драфтов и итоговое ускорение (Speedup).


#### Проверим квантизацию Bitsandbytes на модели из Transformers

In [19]:
# Конфигурация для 4-bit квантизации
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Включаем 4-bit квантизацию
    load_4bit_use_double_quant=True,      # Двойная квантизация для лучшего качества
    bnb_4bit_quant_type="nf4",            # Тип квантизации NF4
    bnb_4bit_compute_dtype=torch.bfloat16 # Тип данных для вычислений
)

# Загружаем квантизованную модель напрямую из Hugging Face
draft_model_4b = AutoModelForCausalLM.from_pretrained(
    DRAFT_ID,
    quantization_config=bnb_config,       # Применяем квантизацию
    device_map="auto",
    attn_implementation="sdpa"
)
draft_model_4b.to(device).eval()

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 896)
    (layers): ModuleList(
      (0-23): 24 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear4bit(in_features=896, out_features=896, bias=True)
          (k_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (v_proj): Linear4bit(in_features=896, out_features=128, bias=True)
          (o_proj): Linear4bit(in_features=896, out_features=896, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (up_proj): Linear4bit(in_features=896, out_features=4864, bias=False)
          (down_proj): Linear4bit(in_features=4864, out_features=896, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((896,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((896,), eps=1e

Линейные слои подменились - квантизация сработала.

In [20]:
# Запускаем бенчмарк
results = benchmark_speculative_sampling(draft_model_4b, target_model, tokenizer, results, prefix_text="NanoQwen_4b")

Running Benchmark... NanoQwen_4b
        Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0     NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1     NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2     NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3  NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4  NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5  NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65
6  NanoQwen_4b  Write a Python funct      12.89       11.42     0.89             0.63        41       65
7  NanoQwen_4b  The capital of Franc      12.89        2.75     0.21             0.00         0       10
8  NanoQwen_4b  Explai

Интересный результат. Квантизация только замедлила генерацию.

Тогда сравним скорость генерации Draft моделей и Target.

In [21]:
# Сравнение скорости генерации
def benchmark_draft_generation_speed(models_dict, tokenizer, num_tokens=50, num_runs=5):
    """
    Сравнивает скорость генерации draft токенов между разными моделями.

    Args:
        models_dict: dict с названиями моделей и самими моделями
        tokenizer: токенизатор
        num_tokens: сколько токенов генерировать
        num_runs: сколько прогонов для усреднения
    """
    results = []

    # Тестовый промпт
    test_prompt = "Write a Python function to calculate Fibonacci numbers."
    input_ids = tokenizer(test_prompt, return_tensors="pt").input_ids.to(device)

    for model_name, model in models_dict.items():
        print(f"Тестируем модель: {model_name}")

        # Прогрев
        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            current_ids = input_ids.clone()
            for _ in range(num_tokens):
                # Генерируем следующий токен
                outputs = model(current_ids)
                logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                current_ids = torch.cat([current_ids, next_token], dim=1)

                if next_token.item() == tokenizer.eos_token_id:
                    break

        # Основной забег
        times = []
        for run in range(num_runs):
            torch.cuda.empty_cache()  # Очищаем кэш GPU

            start_time = time.time()

            with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                current_ids = input_ids.clone()

                for _ in range(num_tokens):
                    # Генерируем следующий токен
                    outputs = model(current_ids)
                    logits = outputs.logits if hasattr(outputs, 'logits') else outputs
                    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
                    current_ids = torch.cat([current_ids, next_token], dim=1)

                    if next_token.item() == tokenizer.eos_token_id:
                        break

            times.append(time.time() - start_time)

        avg_time = sum(times) / len(times)
        tokens_per_sec = num_tokens / avg_time

        results.append({
            'Model': model_name,
            'Avg Time (s)': avg_time,
            'Tokens/sec': tokens_per_sec
        })

    return pd.DataFrame(results)


# Словарь моделей для сравнения
models_to_compare = {
    'Target (fp16)': target_model,
    'NanoQwen (fp16)': draft_model,
    'NanoQwen_HF (fp16)': draft_model_hf,
    'NanoQwen_4bit': draft_model_4b
}

# Запускаем сравнение
speed_results = benchmark_draft_generation_speed(models_to_compare, tokenizer, num_tokens=20, num_runs=3)
print("\n" + "="*50)
print("РЕЗУЛЬТАТЫ СРАВНЕНИЯ СКОРОСТИ ГЕНЕРАЦИИ")
print("="*50)
print(speed_results.to_string(index=False, float_format="{:.2f}".format))


Тестируем модель: Target (fp16)
Тестируем модель: NanoQwen (fp16)
Тестируем модель: NanoQwen_HF (fp16)
Тестируем модель: NanoQwen_4bit

РЕЗУЛЬТАТЫ СРАВНЕНИЯ СКОРОСТИ ГЕНЕРАЦИИ
             Model  Avg Time (s)  Tokens/sec
     Target (fp16)          0.55       36.32
   NanoQwen (fp16)          0.45       44.13
NanoQwen_HF (fp16)          0.47       42.41
     NanoQwen_4bit          1.12       17.82


#### Выводы из теста:

1. Самописная NanoQwen (fp16) оказалась самой быстрой - ~44 tokens/sec
2. HF версия даже немного медленнее - ~44 tokens/sec
3. 4-bit модель сильно медленнее - всего ~18 tokens/sec и Acceptance Rate хуже

Квантизация BNB делается через хуки. На такой маленькой модели, которая сама работает быстро, оверхед поглотил всю выгоду от квантизации.

В текущем сетапе **не стоит использовать 4-bit квантизацию**, так как:
- Замедление генерации draft токенов съедает весь потенциальный выигрыш
- Acceptance rate может упасть из-за снижения качества предсказаний
- Вместо ускорения получили замедление


### 2. LoRA Distillation (1 балл)

Представьте, что Draft модель плохо согласована с Target моделью. Попробуйте дообучить (Fine-Tune) Draft модель на наборе ответов Target модели, используя LoRA (Low-Rank Adaptation). Даже 500 примеров и 10-15 минут обучения на T4 могут повысить Acceptance Rate.


##### Подготовка обучающего датасета

**Создал промпты** при помощи Perplexity, сохранил в файл `train_prompts.json`

In [22]:
# подготовка данных для LoRA Distillation
print("Загружаем промпты из файла...")
with open('train_prompts.json', 'r', encoding='utf-8') as f:
    train_prompts = json.load(f)
print(f"Загружено {len(train_prompts)} промптов")


Загружаем промпты из файла...
Загружено 500 промптов


In [23]:
# Генерируем ответы от Target модели
def generate_target_responses(prompts, model, tokenizer, max_new_tokens=256, batch_size=8):
    """
    Генерирует токены от Target модели на заданные промпты.
    Будут использованы для вычисления loss функции при обучении LoRA адаптера.
    """
    responses = []
    
    print(f"Генерируем ответы от Target модели на {len(prompts)} промптов...")
    
    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i+batch_size]
        batch_messages = []
        
        # Создаем сообщения для батча
        for prompt in batch_prompts:
            messages = [
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt},
            ]
            batch_messages.append(messages)
        
        # Токенизируем батч
        batch_texts = [tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        ) for messages in batch_messages]
        
        inputs = tokenizer(batch_texts, return_tensors="pt", padding=True).to(device)
        
        with torch.no_grad(), torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=0.7,
                top_p=0.9,
                pad_token_id=tokenizer.eos_token_id
            )
        
        for j, output in enumerate(outputs):
            # Находим конец входного текста для каждого примера в батче
            input_length = inputs.input_ids[j].shape[0]
            response = tokenizer.decode(output[input_length:], skip_special_tokens=True)
            responses.append(response)

    return responses

train_responses = generate_target_responses(train_prompts, target_model, tokenizer, max_new_tokens=256, batch_size=100)
train_data = [{"prompt": p, "response": r} for p, r in zip(train_prompts, train_responses)]
with open('train_data.json', 'w', encoding='utf-8') as f:
    json.dump(train_data, f, ensure_ascii=False, indent=2)
print("\nТренировочный датасет сохранен в train_data.json")

Генерируем ответы от Target модели на 500 промптов...


100%|██████████| 5/5 [00:49<00:00,  9.94s/it]


Тренировочный датасет сохранен в train_data.json


In [24]:
train_data = json.load(open('train_data.json', 'r', encoding='utf-8'))
MAX_LENGTH = max([len(train_data[i]['response']) for i in range(len(train_data))])
print(f"Всего примеров: {len(train_data)}")
print(f"Максимальная длина ответа: {MAX_LENGTH}")

Всего примеров: 500
Максимальная длина ответа: 1619


In [25]:
# Создаем датасет
dataset = Dataset.from_list(train_data)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,  # не используем masked language modeling
    pad_to_multiple_of=None,  # не делаем дополнительный padding
)

def format_function(examples):
    # Форматируем промпт
    for i in range(len(examples["prompt"])):
        message = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": examples["prompt"][i]},
        ]
        prompt = tokenizer.apply_chat_template(
            message, tokenize=False, add_generation_prompt=True
        )
        examples["prompt"][i] = prompt
    return examples


# Функция для токенизации датасета
def tokenize_function(examples):
    # Токенизируем промпт и ответ
    tokenized = tokenizer(
        examples["prompt"],
        truncation=False,
        padding=True,
    )
    tokenized["labels"] = tokenizer(
        examples["response"], 
        truncation=True, 
        max_length=MAX_LENGTH, 
        padding=True).input_ids
    return tokenized

# Форматируем датасет
formatted_dataset = dataset.map(format_function, batched=True)

# Токенизируем датасет
tokenized_dataset = formatted_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=formatted_dataset.column_names  # удаляем все исходные поля
)


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

#### Создание LoRA адаптера

In [26]:
# Устанавливаем библиотеку PEFT для работы с LoRA
# !pip install -q peft

# Настройка  LoRA
lora_config = LoraConfig(
    r=8,  # уменьшаем rank для лучшей генерализации
    lora_alpha=16,  # уменьшаем alpha (alpha/r = 2)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # attention layers
        "gate_proj", "up_proj", "down_proj"     # MLP layers
    ],
    lora_dropout=0.1,  # увеличиваем dropout для регуляризации
    bias="none",  # не обучаем bias в LoRA
    task_type="CAUSAL_LM", 
    modules_to_save=None,  # сохраняем только LoRA веса
)

print("Конфигурация LoRA:")
print(f"  - r (rank): {lora_config.r}")
print(f"  - lora_alpha: {lora_config.lora_alpha}")
print(f"  - target_modules: {lora_config.target_modules}")
print(f"  - lora_dropout: {lora_config.lora_dropout}")


# Конфигурация для 4-bit квантизации
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                    # Включаем 4-bit квантизацию
    load_4bit_use_double_quant=True,      # Двойная квантизация для лучшего качества
    bnb_4bit_quant_type="nf4",            # Тип квантизации NF4
    bnb_4bit_compute_dtype=torch.bfloat16 # Тип данных для вычислений
)

# Загружаем квантизованную модель напрямую из Hugging Face
draft_model_lora = AutoModelForCausalLM.from_pretrained(
    DRAFT_ID,
    quantization_config=bnb_config,       # Применяем квантизацию
    device_map="auto",
    attn_implementation="sdpa"
)
draft_model_lora.to(device).eval()

draft_model_lora = PeftModelForCausalLM(draft_model_lora, lora_config)
draft_model_lora.print_trainable_parameters()


Конфигурация LoRA:
  - r (rank): 8
  - lora_alpha: 16
  - target_modules: {'v_proj', 'k_proj', 'up_proj', 'o_proj', 'gate_proj', 'down_proj', 'q_proj'}
  - lora_dropout: 0.1
trainable params: 4,399,104 || all params: 498,431,872 || trainable%: 0.8826


In [27]:
# Дообучение draft-модели с LoRA-адаптером

training_args = TrainingArguments(
    output_dir="./lora_distillation_v3",
    num_train_epochs=20, 
    per_device_train_batch_size=8, 
    gradient_accumulation_steps=8,  # эффективный батч = 64
    learning_rate=1e-4,  # консервативный learning rate
    weight_decay=0.01,
    warmup_steps=5,
    logging_steps=2,
    save_steps=100,
    save_total_limit=2,
    eval_strategy="no",  # без валидации для простоты
    fp16=True,  # используем mixed precision
    dataloader_num_workers=0,  # для совместимости с Jupyter
    remove_unused_columns=False,
    report_to="none",  # отключаем wandb/tensorboard
)

# Создаем Trainer
trainer = Trainer(
    model=draft_model_lora,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

print("Обучение LoRA адаптера ...")
print(f"Количество эпох: {training_args.num_train_epochs}")
print(f"Эффективный батч: {training_args.gradient_accumulation_steps * training_args.per_device_train_batch_size}")
trainer.train()


Обучение LoRA адаптера ...
Количество эпох: 20
Эффективный батч: 64


Step,Training Loss
2,4.714700
4,4.337500
6,3.482900
8,2.767500
10,2.363500
12,1.930100
14,1.546800
16,1.228400
18,0.948300
20,0.795400


'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 513337a9-3c3c-4c56-b7fc-00d0bed13091)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B-Instruct/resolve/main/config.json
Retrying in 1s [Retry 1/5].


TrainOutput(global_step=160, training_loss=0.4990365104749799, metrics={'train_runtime': 305.7896, 'train_samples_per_second': 32.702, 'train_steps_per_second': 0.523, 'total_flos': 717348487680000.0, 'train_loss': 0.4990365104749799, 'epoch': 20.0})

In [28]:
# Запускаем бенчмарк
results = benchmark_speculative_sampling(draft_model_lora, target_model, tokenizer, results, prefix_text="NanoQwen_4b_LoRA_e20")

Running Benchmark... NanoQwen_4b_LoRA_e20
                  Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0               NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1               NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2               NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3            NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4            NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5            NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65
6            NanoQwen_4b  Write a Python funct      12.89       11.42     0.89             0.63        41       65
7            NanoQwen_4b  The capital 

In [29]:
print(generate_simple(draft_model_lora, formatted_dataset[1]['prompt'], 256))

system
You are a helpful assistant.
user
What are the main advantages and disadvantages of artificial intelligence?
assistant
Advantages:
1. Efficiency: Artificial intelligence can automate complex tasks and improve efficiency.
2. Personalization: AI can learn from data and provide personalized recommendations.
3. Cost savings: AI can automate repetitive tasks and reduce costs.
4. Innovation: AI can drive innovation and change the way we live.
5. Entertainment: AI can provide entertainment through virtual reality.

Disadvantages:
1. Job loss: AI can automate jobs and replace humans.
2. Privacy: AI collects data and may collect sensitive information.
3. Bias: AI may be biased and may not work well for everyone.
4. Cybersecurity: AI can be vulnerable to cyber attacks.
5. Ethical: AI can be used for unethical purposes.


Квантизованная модель с LoRA не показала выдающихся результатов, хотя генерирует прилично.


### 3. Layer Pruning Distillation (1.5 балла)

Возьмите Target модель (1.5B) и создайте из нее Draft модель, просто удалив половину слоев (например, каждый второй). Получится "покалеченная" модель ~0.8B. Затем проведите Knowledge Distillation (обучите её восстанавливать логиты оригинальной модели) и используйте результат как Draft модель.


In [30]:
import torch.nn.utils.prune as prune
# Загружаем квантизованную модель напрямую из Hugging Face
target_pruned = AutoModelForCausalLM.from_pretrained(
    TARGET_ID,
    device_map="auto",
    attn_implementation="sdpa"
)
target_pruned.to(device).eval()


# Применяем прунинг к линейным слоям 
def prune_model_20p(model, amount=0.2): 
    for m in model.modules():
        if isinstance(m, nn.Linear):
            prune.ln_structured(m, 'weight', n=2, amount=amount, dim=1)
    return model

In [31]:
def train_pruned_model(model_pruned, epochs=2):
    # Настройка аргументов обучения
    training_args_pruned = TrainingArguments(
        num_train_epochs=epochs,  # оптимальное количество эпох
        per_device_train_batch_size=2,  # маленький батч для стабильности
        gradient_accumulation_steps=4,  # эффективный батч = 8
        learning_rate=1e-4,  # консервативный learning rate
        weight_decay=0.01,
        warmup_steps=5,
        logging_steps=10,
        save_steps=100,
        save_total_limit=2,
        eval_strategy="no",  # без валидации для простоты
        fp16=True,  # используем mixed precision
        dataloader_num_workers=0,  # для совместимости с Jupyter
        remove_unused_columns=False,
        report_to="none",  # отключаем wandb/tensorboard
    )

    # Создаем Trainer
    trainer_pruned = Trainer(
        model=model_pruned,
        args=training_args_pruned,
        train_dataset=tokenized_dataset,
        data_collator=data_collator,
    )

    trainer_pruned.train()
    
    return model_pruned


In [32]:
torchinfo.summary(target_pruned, show_trainable=True)

Layer (type:depth-idx)                             Param #
Qwen2ForCausalLM                                   --
├─Qwen2Model: 1-1                                  --
│    └─Embedding: 2-1                              233,373,696
│    └─ModuleList: 2-2                             --
│    │    └─Qwen2DecoderLayer: 3-1                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-2                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-3                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-4                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-5                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-6                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-7                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-8                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-9                 46,797,824
│    │    └─Qwen2DecoderLayer: 3-10                46,797,824
│    │    └─Qwen2DecoderLayer: 3-11                46,797,824
│    │    └─Qwen2DecoderLayer: 3-1

In [33]:
# Применяем прунинг к линейным слоям на 10%
target_pruned = prune_model_20p(target_pruned, amount=0.1)
target_pruned = train_pruned_model(target_pruned, epochs=2)
# Запускаем бенчмарк
results = benchmark_speculative_sampling(draft_model_lora, target_model, tokenizer, results, prefix_text="BigQwen_pruned_10")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,11.679900
20,1.527300
30,0.705700
40,0.494500
50,0.455400
60,0.376300
70,0.293100
80,0.295000
90,0.251300
100,0.232700


Running Benchmark... BigQwen_pruned_10
                  Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0               NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1               NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2               NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3            NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4            NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5            NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65
6            NanoQwen_4b  Write a Python funct      12.89       11.42     0.89             0.63        41       65
7            NanoQwen_4b  The capital of 

In [34]:
import torchinfo
torchinfo.summary(target_pruned, show_trainable=True)

Layer (type:depth-idx)                             Param #
Qwen2ForCausalLM                                   --
├─Qwen2Model: 1-1                                  --
│    └─Embedding: 2-1                              233,373,696
│    └─ModuleList: 2-2                             --
│    │    └─Qwen2DecoderLayer: 3-1                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-2                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-3                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-4                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-5                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-6                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-7                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-8                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-9                 42,109,952
│    │    └─Qwen2DecoderLayer: 3-10                42,109,952
│    │    └─Qwen2DecoderLayer: 3-11                42,109,952
│    │    └─Qwen2DecoderLayer: 3-1

In [35]:
# Еще 20% - понятно, что это не 10%+20%=30%, но тем не менее...
target_pruned = prune_model_20p(target_pruned, amount=0.2)
target_pruned = train_pruned_model(target_pruned, epochs=2)
# Запускаем бенчмарк
results = benchmark_speculative_sampling(draft_model_lora, target_model, tokenizer, results, prefix_text="BigQwen_pruned_30")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,0.762900
20,0.380800
30,0.334700
40,0.293800
50,0.304800
60,0.277500
70,0.242700
80,0.235100
90,0.226800
100,0.210800


Running Benchmark... BigQwen_pruned_30
                  Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0               NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1               NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2               NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3            NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4            NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5            NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65
6            NanoQwen_4b  Write a Python funct      12.89       11.42     0.89             0.63        41       65
7            NanoQwen_4b  The capital of 

In [36]:
# Еще 20%
target_pruned = prune_model_20p(target_pruned, amount=0.2)
target_pruned = train_pruned_model(target_pruned, epochs=2)
# Запускаем бенчмарк
results = benchmark_speculative_sampling(draft_model_lora, target_model, tokenizer, results, prefix_text="BigQwen_pruned_50")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,0.727100
20,0.347200
30,0.319100
40,0.271000
50,0.260100
60,0.255600
70,0.215700
80,0.221900
90,0.212600
100,0.200400


Running Benchmark... BigQwen_pruned_50
                  Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0               NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1               NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2               NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3            NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4            NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5            NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65
6            NanoQwen_4b  Write a Python funct      12.89       11.42     0.89             0.63        41       65
7            NanoQwen_4b  The capital of 

In [ ]:
torchinfo.summary(target_pruned, show_trainable=True)

Layer (type:depth-idx)                             Param #
Qwen2ForCausalLM                                   --
├─Qwen2Model: 1-1                                  --
│    └─Embedding: 2-1                              233,373,696
│    └─ModuleList: 2-2                             --
│    │    └─Qwen2DecoderLayer: 3-1                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-2                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-3                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-4                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-5                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-6                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-7                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-8                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-9                 26,963,456
│    │    └─Qwen2DecoderLayer: 3-10                26,963,456
│    │    └─Qwen2DecoderLayer: 3-11                26,963,456
│    │    └─Qwen2DecoderLayer: 3-1

In [38]:
# Еще 20%
target_pruned = prune_model_20p(target_pruned, amount=0.2)
target_pruned = train_pruned_model(target_pruned, epochs=2)
# Запускаем бенчмарк
results = benchmark_speculative_sampling(draft_model_lora, target_model, tokenizer, results, prefix_text="BigQwen_pruned_70")

The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
10,0.718100
20,0.324300
30,0.262200
40,0.249900
50,0.265600
60,0.248900
70,0.216200
80,0.214100
90,0.210200
100,0.196400


Running Benchmark... BigQwen_pruned_70
                  Prefix                Prompt  Std (t/s)  Spec (t/s)  Speedup  Acceptance Rate  Accepted  Drafted
0               NanoQwen  Write a Python funct      12.69       30.98     2.44             1.00        45       45
1               NanoQwen  The capital of Franc      12.92       24.16     1.87             1.00         2        2
2               NanoQwen  Explain the theory o      12.93       20.80     1.61             0.60        39       65
3            NanoQwen_HF  Write a Python funct      12.88       30.51     2.37             1.00        45       45
4            NanoQwen_HF  The capital of Franc      12.92       23.58     1.83             1.00         2        2
5            NanoQwen_HF  Explain the theory o      12.93       20.35     1.57             0.60        39       65
6            NanoQwen_4b  Write a Python funct      12.89       11.42     0.89             0.63        41       65
7            NanoQwen_4b  The capital of 

In [39]:
torchinfo.summary(target_pruned, show_trainable=True)

Layer (type:depth-idx)                             Param #
Qwen2ForCausalLM                                   --
├─Qwen2Model: 1-1                                  --
│    └─Embedding: 2-1                              233,373,696
│    └─ModuleList: 2-2                             --
│    │    └─Qwen2DecoderLayer: 3-1                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-2                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-3                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-4                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-5                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-6                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-7                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-8                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-9                 21,572,096
│    │    └─Qwen2DecoderLayer: 3-10                21,572,096
│    │    └─Qwen2DecoderLayer: 3-11                21,572,096
│    │    └─Qwen2DecoderLayer: 3-1

#### Вывод

Пруненная "большая" модель, показывает некоторое ухудшение Acceptance Rate,
но при этом показывает небольшое (очень небольшое) ускорение генерации.